In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor
from utils import create_excel
kl_data = pd.read_excel('../20240607FULL.xlsx', index_col = 0)

X = kl_data.drop(['惯习面', '基面直径（um）', '基面长度（um）',
                               '基面厚度（nm）', '基面析出相体积分数%', '柱面相直径（nm）', '柱面相长度（nm）',
                               '柱面相宽度（nm）', '柱面析出相体积分数%', '析出相分布','屈服强度', '抗拉强度 （UTS）','追踪编号'],axis = 1)
print('总共有{}个特征'.format(X.shape[1]))
y = kl_data['屈服强度']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
    model = RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits

# 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]

# 输出到新的表格中
importance_output_file = 'average_feature_importance_qf.xlsx'
create_excel(importance_output_file)
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name='qf_279features',index=False)
    print(f'文件保存到{importance_output_file}中')

scores_file= 'scores_file.xlsx'
create_excel(scores_file)
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name='qf_279eatures', index=False)
print(f'分数文件已保存到{scores_file}文件中')



总共有280个特征
[386, 442, 420, 448, 139, 154, 289, 244, 427, 410, 475, 543, 455, 484, 256, 334, 300, 358, 360, 365, 320, 255, 266, 184, 218, 160, 240, 290, 300, 285, 194, 201, 190, 169, 116, 222, 241, 184, 218, 221, 223, 232, 288, 414, 471, 543, 360, 169, 132, 184, 176, 184, 176]
<class 'pandas.core.series.Series'>
55.60000000000002
102.95999999999998
5.819999999999993
105.41000000000003
49.910000000000025
11.04333333333335
6.304166666666646
6.723333333333329
76.77999999999997
67.80000000000001
52.584857142857146
<class 'pandas.core.series.Series'>
14.870000000000005
39.129999999999995
103.38999999999999
11.810000000000002
14.203333333333376
12.939999999999998
69.78999999999999
2.6850000000000023
81.07
35.66552380952379
35.66552380952379
<class 'pandas.core.series.Series'>
2.980000000000018
42.01999999999998
31.350000000000023
78.78
13.129999999999995
14.939999999999998
35.47333333333336
67.98666666666668
102.25999999999999
75.43
46.03
<class 'pandas.core.series.Series'>
8.910000000000025
1

In [25]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor
from utils import create_excel
qf_data = pd.read_excel('../20240607FULL.xlsx', index_col = 0)

X = qf_data.drop(['屈服强度', '抗拉强度 （UTS）','追踪编号'],axis = 1)
print('总共有{}个特征'.format(X.shape[1]))
y = qf_data['屈服强度']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
    model = RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits

# 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]

# 输出到新的表格中
importance_output_file = 'average_feature_importance_qf.xlsx'
create_excel(importance_output_file)
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name='qf_289features',index=False)
    print(f'文件保存到{importance_output_file}中')

scores_file= 'scores_file.xlsx'
create_excel(scores_file)
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name='qf_289eatures', index=False)
print(f'分数文件已保存到{scores_file}文件中')



总共有290个特征
[386, 442, 420, 448, 139, 154, 289, 244, 427, 410, 475, 543, 455, 484, 256, 334, 300, 358, 360, 365, 320, 255, 266, 184, 218, 160, 240, 290, 300, 285, 194, 201, 190, 169, 116, 222, 241, 184, 218, 221, 223, 232, 288, 414, 471, 543, 360, 169, 132, 184, 176, 184, 176]
<class 'pandas.core.series.Series'>
58.360000000000014
67.30000000000001
4.089999999999975
110.54000000000002
60.0
11.189999999999998
4.289999999999992
7.0800000000000125
94.48000000000002
55.84
27.50999999999999
<class 'pandas.core.series.Series'>
15.430000000000007
38.84
89.38999999999999
3.8100000000000023
36.68000000000001
12.340000000000003
69.4
0.21999999999999886
78.86000000000001
37.31
37.31
<class 'pandas.core.series.Series'>
13.949999999999989
44.52999999999997
7.699999999999989
74.77000000000001
10.780000000000001
23.80000000000001
47.22999999999999
88.5
101.68
48.69999999999999
26.159999999999997
<class 'pandas.core.series.Series'>
12.519999999999982
23.72999999999999
56.54000000000002
14.16000000000002

In [26]:
import pandas as pd 
pd.read_excel('../20240607Feature.xlsx')
important_features=pd.read_excel('average_feature_importance_qf.xlsx',sheet_name='qf_289features')
top_feature=important_features['Feature'].head(15).to_list()
print(top_feature)

['MagpieData maximum MeltingT', 'Zr fraction', 'MagpieData range MeltingT', 'Grain Size', 'Lambda entropy', 'MagpieData range GSvolume_pa', 'MagpieData mean GSmagmom', 'MagpieData range GSmagmom', 'Shear modulus strength model', 'MagpieData minimum GSvolume_pa', 'Shear modulus local mismatch', 'Shear modulus delta', 'Interant electrons', 'Calculated Grain Boundary', 'Radii gamma']


原始Top10特征

In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='qf_22'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['屈服强度']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_qf.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[386, 442, 420, 448, 139, 154, 289, 244, 427, 410, 475, 543, 455, 484, 256, 334, 300, 358, 360, 365, 320, 255, 266, 184, 218, 160, 240, 290, 300, 285, 194, 201, 190, 169, 116, 222, 241, 184, 218, 221, 223, 232, 288, 414, 471, 543, 360, 169, 132, 184, 176, 184, 176]
<class 'pandas.core.series.Series'>
55.089999999999975
70.68
5.949999999999989
122.38999999999999
72.67000000000002
12.5
10.300000000000011
15.77000000000001
67.43
42.81
28.180000000000007
<class 'pandas.core.series.Series'>
5.009999999999991
53.66
87.83999999999997
1.8600000000000136
34.339999999999975
16.27000000000001
78.19999999999999
7.8799999999999955
71.94
59.19999999999999
59.19999999999999
<class 'pandas.core.series.Series'>
60.69
73.74000000000001
5.029999999999973
73.4
10.580000000000013
21.370000000000005
57.19
30.430000000000007
83.43
11.920000000000016
22.52000000000001
<class 'pandas.core.series.Series'>
35.35000000000002
31.409999999999997
17.439999999999998
2.0600000000000023
15.920000000000016
9.65999999999

,Feature,Importance
0,MagpieData range MeltingT,0.473503
1,MagpieData maximum MeltingT,0.278238
2,Grain Size,0.076252
3,Calculated Grain Boundary,0.032058
4,Calculated Yield Strength,0.025721
5,柱面析出相体积分数%,0.014617
6,mean AtomicRadius,0.013607
7,基面析出相体积分数%,0.013128
8,Calculated Solid Solution,0.011507
9,基面厚度（nm）,0.009308


文件保存到average_feature_importance_qf.xlsx中
分数文件已保存到scores_file.xlsx文件中


Top5特征

In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='qf_27'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['屈服强度']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_qf.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[386, 442, 420, 448, 139, 154, 289, 244, 427, 410, 475, 543, 455, 484, 256, 334, 300, 358, 360, 365, 320, 255, 266, 184, 218, 160, 240, 290, 300, 285, 194, 201, 190, 169, 116, 222, 241, 184, 218, 221, 223, 232, 288, 414, 471, 543, 360, 169, 132, 184, 176, 184, 176]
<class 'pandas.core.series.Series'>
52.22000000000003
67.07
14.310000000000002
121.64999999999998
74.68
18.310000000000002
4.069999999999993
13.240000000000009
59.389999999999986
45.46000000000001
27.53
<class 'pandas.core.series.Series'>
10.029999999999973
47.41
94.55000000000001
5.660000000000025
33.18000000000001
14.800000000000011
78.75
6.719999999999999
77.73000000000002
45.94999999999999
45.94999999999999
<class 'pandas.core.series.Series'>
10.470000000000027
52.24000000000001
10.160000000000025
64.50999999999999
1.1100000000000136
17.889999999999986
58.34
29.090000000000003
89.24000000000001
36.26999999999998
26.28
<class 'pandas.core.series.Series'>
19.529999999999973
32.599999999999994
53.54000000000002
14.660000000

,Feature,Importance
0,MagpieData range MeltingT,0.282211
1,Interant d electrons,0.278323
2,MagpieData maximum MeltingT,0.210966
3,Grain Size,0.084010
4,Calculated Grain Boundary,0.019215
5,Calculated Yield Strength,0.016076
6,柱面析出相体积分数%,0.013760
7,基面长度（um）,0.008941
8,mean AtomicRadius,0.008525
9,基面析出相体积分数%,0.008149


文件保存到average_feature_importance_qf.xlsx中
分数文件已保存到scores_file.xlsx文件中


In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='qf_32'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['屈服强度']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_qf.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[386, 442, 420, 448, 139, 154, 289, 244, 427, 410, 475, 543, 455, 484, 256, 334, 300, 358, 360, 365, 320, 255, 266, 184, 218, 160, 240, 290, 300, 285, 194, 201, 190, 169, 116, 222, 241, 184, 218, 221, 223, 232, 288, 414, 471, 543, 360, 169, 132, 184, 176, 184, 176]
<class 'pandas.core.series.Series'>
55.389999999999986
64.30000000000001
12.350000000000023
121.88
70.31
20.120000000000005
1.1699999999999875
11.930000000000007
76.30000000000001
52.18000000000001
28.650000000000006
<class 'pandas.core.series.Series'>
11.160000000000025
52.27000000000001
95.67000000000002
4.189999999999998
33.31
13.439999999999998
74.59
5.969999999999999
75.19
42.24000000000001
42.24000000000001
<class 'pandas.core.series.Series'>
5.560000000000002
44.50999999999999
6.589999999999975
66.28999999999999
2.8400000000000034
22.840000000000003
59.80000000000001
37.93000000000001
89.11000000000001
39.889999999999986
26.860000000000014
<class 'pandas.core.series.Series'>
15.20999999999998
25.669999999999987
58.350

,Feature,Importance
0,MagpieData maximum MeltingT,0.255107
1,MagpieData range MeltingT,0.236817
2,Radii gamma,0.104038
3,Interant d electrons,0.086647
4,Mn fraction,0.085427
5,Grain Size,0.082836
6,MagpieData mean NdValence,0.020080
7,Calculated Grain Boundary,0.017430
8,Calculated Yield Strength,0.013381
9,柱面析出相体积分数%,0.013376


文件保存到average_feature_importance_qf.xlsx中
分数文件已保存到scores_file.xlsx文件中


In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='qf_16'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['屈服强度']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_qf.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[386, 442, 420, 448, 139, 154, 289, 244, 427, 410, 475, 543, 455, 484, 256, 334, 300, 358, 360, 365, 320, 255, 266, 184, 218, 160, 240, 290, 300, 285, 194, 201, 190, 169, 116, 222, 241, 184, 218, 221, 223, 232, 288, 414, 471, 543, 360, 169, 132, 184, 176, 184, 176]
<class 'pandas.core.series.Series'>
46.839999999999975
62.410000000000025
12.730000000000018
130.36
80.82999999999998
15.039999999999992
9.050000000000011
9.97999999999999
53.97000000000003
39.5
28.830000000000013
<class 'pandas.core.series.Series'>
5.180000000000007
56.47
80.10000000000002
4.600000000000023
30.220000000000027
11.180000000000007
78.4
9.050000000000011
70.50999999999999
51.44
51.44
<class 'pandas.core.series.Series'>
32.629999999999995
74.89999999999998
5.069999999999993
65.94999999999999
4.039999999999992
9.900000000000006
64.0
28.159999999999997
80.71000000000001
19.430000000000007
19.78
<class 'pandas.core.series.Series'>
19.54000000000002
41.49000000000001
11.310000000000002
1.5400000000000205
28.99000000

,Feature,Importance
0,MagpieData range MeltingT,0.437752
1,MagpieData maximum MeltingT,0.334306
2,Grain Size,0.096803
3,Calculated Yield Strength,0.038507
4,柱面析出相体积分数%,0.022153
5,基面厚度（nm）,0.012505
6,基面析出相体积分数%,0.010858
7,基面长度（um）,0.009882
8,柱面相长度（nm）,0.009654
9,基面直径（um）,0.008909


文件保存到average_feature_importance_qf.xlsx中
分数文件已保存到scores_file.xlsx文件中


In [31]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='qf_18'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['屈服强度']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_qf.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[386, 442, 420, 448, 139, 154, 289, 244, 427, 410, 475, 543, 455, 484, 256, 334, 300, 358, 360, 365, 320, 255, 266, 184, 218, 160, 240, 290, 300, 285, 194, 201, 190, 169, 116, 222, 241, 184, 218, 221, 223, 232, 288, 414, 471, 543, 360, 169, 132, 184, 176, 184, 176]
<class 'pandas.core.series.Series'>
52.19999999999999
65.39999999999998
8.410000000000025
130.45
80.45999999999998
15.280000000000001
12.740000000000009
14.759999999999991
57.81
35.16999999999999
27.539999999999992
<class 'pandas.core.series.Series'>
8.149999999999977
55.03999999999999
88.56
12.629999999999995
29.970000000000027
16.03
74.12
8.710000000000008
70.92000000000002
53.870000000000005
53.870000000000005
<class 'pandas.core.series.Series'>
45.45999999999998
81.52999999999997
7.949999999999989
67.46000000000001
1.9499999999999886
22.789999999999992
57.09
34.71000000000001
81.66999999999999
15.149999999999977
21.379999999999995
<class 'pandas.core.series.Series'>
31.089999999999975
26.24000000000001
21.430000000000007

,Feature,Importance
0,MagpieData range MeltingT,0.383792
1,MagpieData maximum MeltingT,0.375566
2,Grain Size,0.078332
3,Calculated Grain Boundary,0.030758
4,Calculated Yield Strength,0.030114
5,柱面析出相体积分数%,0.016742
6,Calculated Solid Solution,0.015999
7,基面析出相体积分数%,0.012602
8,基面厚度（nm）,0.012216
9,基面长度（um）,0.009845


文件保存到average_feature_importance_qf.xlsx中
分数文件已保存到scores_file.xlsx文件中


含析出相top15特征

In [32]:
# GBoost效果不如RF